# 00 · Setup: Loja Virtual no SQL Server (origem)

**Trabalho 3 — Lakehouse com Databricks e arquitetura medalhão** · Autores: **Murilo Mandelli** e **Augusto Benedetti**

Roda **no computador (WSL)**, não no Databricks. Cria o banco **`LojaDB`** no SQL Server (Docker)
com as **5 tabelas** da loja virtual — o mesmo cenário do Trabalho 2 — a partir de `data/origem/`.

Os dados de origem têm **problemas de qualidade inseridos de propósito** (e-mail em maiúsculas,
CPF com máscara, cliente duplicado, preço negativo, quantidade zero…), como em um sistema real.
Eles serão corrigidos na camada **Silver**.

**Pré-requisito:** `uv run task up` (sobe o SQL Server na porta 1434).

In [1]:
import pandas as pd

from lakehouse_databricks import TABELAS, conectar_sqlserver
from lakehouse_databricks.config import DB_DATABASE, DB_PORT, DB_SERVER, PASTA_ORIGEM

print(f"Servidor: {DB_SERVER}:{DB_PORT} | Banco: {DB_DATABASE}")
conn_master = conectar_sqlserver()
print(conn_master.cursor().execute("SELECT @@VERSION").fetchone()[0].splitlines()[0])

Servidor: localhost:1434 | Banco: LojaDB


Microsoft SQL Server 2025 (RTM-CU9-GDR) (KB5126967) - 17.0.5015.4 (X64) 


## 1. Criar o banco `LojaDB` (recriado a cada execução)

In [2]:
cur = conn_master.cursor()
cur.execute(f"""
IF DB_ID('{DB_DATABASE}') IS NOT NULL
BEGIN
    ALTER DATABASE [{DB_DATABASE}] SET SINGLE_USER WITH ROLLBACK IMMEDIATE;
    DROP DATABASE [{DB_DATABASE}];
END
""")
cur.execute(f"CREATE DATABASE [{DB_DATABASE}]")
conn_master.close()

conn = conectar_sqlserver(DB_DATABASE)
cursor = conn.cursor()
print(f"Banco [{DB_DATABASE}] criado.")

Banco [LojaDB] criado.


## 2. DDL — as 5 tabelas

In [3]:
DDL = [
    """CREATE TABLE categoria (
        id_categoria  INT           NOT NULL PRIMARY KEY,
        nome          VARCHAR(50)   NOT NULL,
        descricao     VARCHAR(200)  NULL
    )""",
    """CREATE TABLE cliente (
        id_cliente   INT           NOT NULL PRIMARY KEY,
        nome         VARCHAR(100)  NOT NULL,
        email        VARCHAR(120)  NOT NULL,
        cpf          VARCHAR(14)   NOT NULL,
        cidade       VARCHAR(80)   NULL,
        uf           CHAR(2)       NULL,
        dt_cadastro  DATE          NOT NULL
    )""",
    """CREATE TABLE produto (
        id_produto    INT            NOT NULL PRIMARY KEY,
        id_categoria  INT            NOT NULL REFERENCES categoria(id_categoria),
        nome          VARCHAR(120)   NOT NULL,
        preco         DECIMAL(10,2)  NOT NULL,
        estoque       INT            NOT NULL,
        ativo         BIT            NOT NULL
    )""",
    """CREATE TABLE pedido (
        id_pedido    INT            NOT NULL PRIMARY KEY,
        id_cliente   INT            NOT NULL REFERENCES cliente(id_cliente),
        dt_pedido    DATETIME2(0)   NOT NULL,
        status       VARCHAR(20)    NOT NULL,
        valor_total  DECIMAL(12,2)  NOT NULL
    )""",
    """CREATE TABLE item_pedido (
        id_item         INT            NOT NULL PRIMARY KEY,
        id_pedido       INT            NOT NULL REFERENCES pedido(id_pedido),
        id_produto      INT            NOT NULL REFERENCES produto(id_produto),
        quantidade      INT            NOT NULL,
        preco_unitario  DECIMAL(10,2)  NOT NULL
    )""",
]
for comando in DDL:
    cursor.execute(comando)
print("5 tabelas criadas.")

5 tabelas criadas.


## 3. Carga dos CSV de `data/origem/`

In [4]:
cursor.fast_executemany = True

for tabela in TABELAS:
    df = pd.read_csv(PASTA_ORIGEM / f"{tabela}.csv", dtype={"cpf": str, "uf": str}, keep_default_na=False, na_values=[""])
    colunas = ", ".join(df.columns)
    marcadores = ", ".join("?" for _ in df.columns)
    linhas = [
        tuple(None if pd.isna(v) else (v.item() if hasattr(v, "item") else v) for v in registro)
        for registro in df.itertuples(index=False)
    ]
    cursor.executemany(f"INSERT INTO {tabela} ({colunas}) VALUES ({marcadores})", linhas)
    print(f"{tabela:<12} {len(linhas):>6} registros")

categoria         8 registros


cliente        1015 registros


produto         200 registros


pedido         3000 registros


item_pedido    8971 registros


## 4. Uma amostra dos problemas de qualidade na origem

In [5]:
cursor.execute("""
SELECT TOP 6 id_cliente, nome, email, cpf, cidade, uf
FROM cliente
WHERE nome LIKE ' %' OR email COLLATE Latin1_General_CS_AS <> LOWER(email) OR cpf LIKE '%.%' OR cidade IS NULL
ORDER BY id_cliente
""")
pd.DataFrame.from_records(cursor.fetchall(), columns=[d[0] for d in cursor.description])

,id_cliente,nome,email,cpf,cidade,uf
0,6,Beatriz Macedo,marcela26@live.com,147.209.358-50,Borges,AC
1,14,eloá caldeira,teixeiraeduarda@yahoo.com.br,18524069767,da Cunha de Minas,AC
2,15,Oliver Albuquerque,britoravy@gmail.com,418.596.273-82,Guerra,MT
3,26,Dr. Thales Guerra,luiz-gustavo37@uol.com.br,596.743.108-00,Lopes,GO
4,28,Maria Luísa Fernandes,rochamaya@icloud.com,83625701995,NaN,PI
5,29,Lucas Gabriel Pacheco,zvasconcelos@live.com,750.916.284-01,Vieira,PB


In [6]:
cursor.execute("SELECT status, COUNT(*) AS qtd FROM pedido GROUP BY status ORDER BY status")
pd.DataFrame.from_records(cursor.fetchall(), columns=["status", "qtd"])

,status,qtd
0,ENVIADO,29
1,CANCELADO,292
2,ENTREGUE,1683
3,ENVIADO,404
4,PAGO,312
5,PENDENTE,280


In [7]:
cursor.close()
conn.close()